# Preprocessing

### Load and Convert Fine-tuned Transformer Model to TransformerLens

In [2]:
from src import load_finetuned_model

base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")

model.eval()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


Moving model to device:  mps


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-23): 24 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

## Find the data with the correct answer

In [ ]:
from src import filter_correct_data
import pandas as pd

dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_corrected.csv"
filtered_data_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered.csv"
test_data = pd.read_csv(dataset_path)

filtered_data = filter_correct_data(model, test_data, "original_sentence", "original_triplet", save_path=filtered_data_path)

## Create EAP Dataset

#### Building the Dataset

In [7]:

from src.utils import build_eap_dataset
import pandas as pd


filtered_data = pd.read_csv(
    "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered.csv")

eap_df = build_eap_dataset(
    model=model,
    df=filtered_data,
    sentence_col="original_sentence",
    triplet_col="original_triplet",
    corrupted_col="counterfact3_modified",
    corrupted_triplet_col="counterfact_triplet3_modified",
    suffix="[S]",
    idx=2,  # 0 for aspect, 1 for opinion, 2 for sentiment,
    filer_same_length_counterfactuals=True
)
eap_df.to_csv("eap_dataset/eap_dataset_sentiment_multitokens.csv", index=False)

Removed 39 out of 53 datapoints that does not match token length.
Filtered data size len(eap_data)=14


# EAP-IG

In [8]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

In [9]:
import argparse
import ast
import os
from functools import partial
from random import random
from typing import Optional

import pandas as pd
import transformers
from torch.utils.data import Dataset, DataLoader
import torch
from typing_extensions import Tuple, List, Union

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline, evaluate_baseline_multitoken, evaluate_graph_multitoken
from eap.attribute import attribute
from src.utils import build_eap_dataset
from src import load_finetuned_model, filter_correct_data
from src.metric import logit_diff

In [10]:
# load model
base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


Moving model to device:  mps


In [11]:
# load dataset
ds = pd.read_csv("eap_dataset/eap_dataset_opinion_multitokens.csv")

In [12]:
g = Graph.from_model(model)

In [13]:
baseline = evaluate_baseline_multitoken(
    model,
    df=ds,
    metrics=[logit_diff],
    run_corrupted=False,
    batch_size=4
)
print(f"Original performance is logit_dif={baseline}")

Evaluating groups:   0%|          | 0/3 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Evaluating groups: 100%|██████████| 3/3 [00:33<00:00, 11.31s/it]

✔ Evaluated 44 total tokens.
Original performance is logit_dif=19.222911834716797


In [14]:
attribute(
    model=model,
    graph=g,
    dataloader=ds,  # can be Dataset or DataLoader
    metric=partial(logit_diff,loss=False, mean=True),
    method="EAP-IG-inputs",
    ig_steps=5,
    is_absa=True,
    batch_size=4,
    device="mps"
)

Token len = 6: 100%|██████████| 2/2 [05:06<00:00, 153.11s/it]


In [15]:
n_edges = g.real_edge_mask.sum().item()  # total 171K edges for qwen2.5-0.5B
for topk in [100, 200, 500, 1000, 2000, 5000, 10000, 20000]:
    g.reset()
    g.apply_topn(topk, True)
    results = evaluate_graph_multitoken(model=model,
                                        graph=g,
                                        df=ds,  # your full dataset DataFrame
                                        metrics=[partial(logit_diff, mean=True, loss=False)],  # or just [logit_diff]
                                        batch_size=4,)

    print(f"with top-k = {topk} ({topk/n_edges:.1%}), the circuit's performance is {results}, faithfulness={results/baseline:.1%}")
    g.to_pt(f'outputs/opinion_circuit_topk-{topk}.pt')

    print(f"included nodes: {g.count_included_nodes()}, included edges: {g.count_included_edges()}")

Evaluating groups: 100%|██████████| 3/3 [00:43<00:00, 14.61s/it]


with top-k = 100 (0.1%), the circuit's performance is -12.78785514831543, faithfulness=-66.5%
included nodes: 22, included edges: 48


Evaluating groups: 100%|██████████| 3/3 [00:42<00:00, 14.06s/it]


with top-k = 200 (0.1%), the circuit's performance is -12.771842956542969, faithfulness=-66.4%
included nodes: 40, included edges: 123


Evaluating groups: 100%|██████████| 3/3 [00:48<00:00, 16.12s/it]


...

with top-k = 5000 (2.8%), the circuit's performance is -12.20823860168457, faithfulness=-63.5%
included nodes: 296, included edges: 4886


Evaluating groups: 100%|██████████| 3/3 [00:42<00:00, 14.27s/it]


with top-k = 10000 (5.6%), the circuit's performance is -12.170953750610352, faithfulness=-63.3%
included nodes: 340, included edges: 9565


Evaluating groups: 100%|██████████| 3/3 [00:45<00:00, 15.10s/it]

with top-k = 20000 (11.1%), the circuit's performance is -11.9398832321167, faithfulness=-62.1%
included nodes: 359, included edges: 19990


# Circuit Merging

In [ ]:
from src.utils import edge_merging

graph_paths = ["outputs/multitokens/aspect_circuit_topk-20000.pt", "outputs/multitokens/sentiment_circuit_topk-20000.pt", "outputs/multitokens/opinion_circuit_topk-20000.pt"]

complete_edges = edge_merging(graph_paths=graph_paths)

complete_edges.to_csv("outputs/multitokens/complete_circuit_topk-20000.csv", index=None)

In [5]:
complete_edges.shape

(59966, 3)